# 2.6 Mini-Project: Personal Task Agent

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

## Learning objectives

- Design an agent from a specification instead of a tutorial: choose the tools, the system prompt, and the reliability stack yourself.
- Build a persistent task manager (add, list, complete, prioritise) backed by a real store, driven by a `create_agent` harness.
- Apply every reliability pattern from lessons 2.4 and 2.5: error handling per tool, retry only on transient failures, budgets on expensive calls, and a hard backstop.
- Evaluate your own agent against an acceptance checklist and a rubric, the way the module assessment will.

## Prerequisites

All of Module 2 so far: tool calling (2.1), custom tools (2.2), the ReAct pattern and `create_agent` (2.3), errors and retries (2.4), and the research-agent lab (2.5). You will reuse the calculator, the error-middleware stack, and the trace-reading habit from the lab.

## The brief

Build a personal task agent: a program the learner runs in the terminal that manages a to-do list through natural language. It must handle a realistic session like this one:

> "Add 'prepare EU AI Act summary' due Friday, high priority. Also add 'buy groceries' with no due date. What is due this week? Mark groceries done. How much would 3 hours of my EUR 45/hour rate cost, with 19% VAT?"

One program, one agent, real persistence: tasks must survive a restart. This is the first thing in the course you build without step-by-step instructions. The specification below is your only guide, which is the point: reading a spec and turning it into a working agent is the actual skill.

## Required tools

Design the toolset yourself, but it must cover these four capabilities. Suggested implementations are in brackets; use them or write your own.

1. **Task store** (JSON file on disk, one task per record: id, title, due date, priority, done flag). Tools: `add_task`, `list_tasks`, `complete_task`, `set_priority`. The store must be real persistence, not an in-memory list: kill the program and the tasks stay.
2. **Current date/time** (lesson 2.2's `get_current_time`). "Due Friday" is meaningless without it; the agent must resolve relative dates itself.
3. **Calculator** (lesson 2.2's, with the divide-by-zero error string). The VAT question must route through it.
4. **Web search** (optional but recommended, from lab 2.5). Useful if the learner asks "what is the EU AI Act deadline for GPAI models?" mid-session. If you include it, it gets the strictest budget.

Do not add tools the brief does not need. Every extra tool is a new failure surface and a new line in the system prompt.

## Reliability requirements (non-negotiable)

These come straight from lessons 2.4 and 2.5, and the rubric scores them:

- **Budgets in code.** `ModelCallLimitMiddleware` on the run, and `ToolCallLimitMiddleware` on the search tool if you include it. Infinite loops are a billing event; prove yours cannot happen.
- **Errors as data for deterministic failures.** Completing a task id that does not exist, adding a task with an unparseable date: return a message the model can act on, not an exception.
- **Retries only for transient failures.** File writes and searches may retry on I/O errors; validation failures must not.
- **A hard backstop.** `recursion_limit` on every invoke, with the `GraphRecursionError` path handled (a clean message, not a stack trace).
- **Honest output.** If a tool fails and the agent cannot complete the request, it says so. No invented task ids, no phantom completions. Read the trace to verify, not just the final answer.

## Suggested structure

You own the architecture, but a shape that works well:

In [ ]:
task_agent.py        # entry point: reads a line, invokes the agent, prints the reply
tools_tasks.py       # the four task tools plus the JSON store
tools_misc.py        # time, calculator (reused from earlier lessons)
agent.py             # create_agent with system prompt + middleware stack

Keep the system prompt short and behavioural: what the agent is, when to use each tool, the honesty rules, and the instruction to resolve relative dates ("Friday" means the upcoming Friday from `get_current_time`). Long system prompts full of formatting rules are a smell; formatting belongs in the final-answer instruction, not in ten bullet points.

## Stretch goals (pick at most two)

- **Undo.** A `undo_last_change` tool backed by a small append-only change log. Teaches you what "state" really means.
- **Recurring tasks.** "Every Monday" creates the next instance on completion. Small feature, surprisingly tricky date logic.
- **Daily briefing.** A `--briefing` flag that prints today's due tasks and overdue ones without starting the chat loop. Teaches you to reuse the tool layer without the agent.
- **Ollama path.** Run the whole agent on a local model via `langchain-ollama` (lesson 1.4). Expect weaker tool calling; document exactly where it breaks. This one teaches more about model capability gaps than any lecture could.

## Acceptance checklist

Run through this list yourself before you consider the project done. Every item must pass.

- [ ] Add three tasks with different priorities and due dates; list shows all three, correctly ordered.
- [ ] Restart the program; all tasks and their state persist.
- [ ] Complete a task by title ("mark groceries done"); the list reflects it.
- [ ] Ask "what is due this week?"; the agent resolves the week from the current date and lists only matching tasks.
- [ ] Ask the VAT question; the agent uses the calculator tool (check the trace) and the arithmetic is right.
- [ ] Complete task id 999 (does not exist); the agent reports the error and offers the real list instead of crashing or inventing.
- [ ] With the network disconnected, the agent still manages tasks; if search is included, it says search is unavailable and continues.
- [ ] A deliberately confusing request ("add task do the thing soon, high-ish priority") gets a clarifying question or a sensible default, not a crash.
- [ ] Inspect one full trace: no repeated identical tool calls, no invented task ids, every claimed action backed by a tool observation.

## Rubric (how this is assessed)

| Criterion | Weight | What excellence looks like |
|---|---|---|
| Working task management | 30% | All acceptance items pass; persistence is real; relative dates resolve correctly. |
| Reliability engineering | 30% | Budgets in code, errors as data, transient-only retries, handled backstop; trace shows no loops or invented actions. |
| Tool design | 20% | Small, honest toolset; docstrings teach the model when to use each tool; no unused or overlapping tools. |
| Code quality | 10% | Clear modules, no dead code, pinned versions in a comment or requirements line, `.env` for keys. |
| Honesty under failure | 10% | Agent discloses what it could not do; error paths tested deliberately, not just the happy path. |

Score yourself honestly against this before the module assessment. If you cannot tick an acceptance item, that is a finding, not a failure: write down what you tried and what you learned.

## What can go wrong

- **Date ambiguity.** "Friday" on a Saturday, "next week", "in a bit": relative dates are the hardest part of this project. Resolve them with explicit rules in the system prompt (e.g. "Friday" means the upcoming Friday; "next week" means Monday of next week) and state the interpretation back to the user.
- **Title matching.** "Mark groceries done" when two tasks contain "groceries" is ambiguous. The robust pattern: list candidates and ask, rather than guessing. Guessing wrong and completing the wrong task is the kind of silent failure users remember.
- **The JSON store corrupts.** A crash mid-write leaves half a file. Write to a temp file and rename (atomic on most systems), and keep the schema simple enough that a human can repair it by hand.
- **Over-tooling.** Six tools for a to-do list is too many. If two tools overlap, the model hesitates or picks randomly. Four focused tools beat eight clever ones.
- **Testing only the happy path.** The rubric's honesty weight exists because untested error paths are where real agents fail. Break your agent on purpose: bad ids, bad dates, no network, duplicate titles.
- **Data on disk, GDPR in mind.** A task list is personal data. If this ever left your laptop (a hosted version, shared logs), the store and any persisted traces would need the same treatment Module 6 teaches: minimisation, redaction, retention limits. For now, keep the file local and out of version control.

## Review questions

**1. Why does the brief require real file persistence instead of an in-memory list?**
Because a task manager that forgets everything on restart is a demo, not a tool. Persistence forces you to design a real store (schema, atomic writes, ids), and it is the first step toward the durable agents Module 4 builds with checkpointers.

**2. Your agent completes the wrong task when two titles are similar. What is the correct behaviour, and which lesson does it come from?**
List the matching candidates and ask the user to choose, rather than guessing. This is the honesty principle from lessons 2.4 and 2.5: a wrong silent action is worse than a clarifying question, and the trace must back every claimed action.

**3. Where do retries belong in this project, and where are they forbidden?**
Retries belong on transient failures: network timeouts on the optional search tool, I/O errors on file writes. They are forbidden on deterministic failures: unknown task ids, unparseable dates, validation errors. Lesson 2.4's rule: correct deterministic failures, retry transient ones.

**4. A learner's agent passes every acceptance item but has no `recursion_limit` and no call middleware. What do you tell them, and why?**
It is not done. The acceptance list tests behaviour; the reliability requirements test what happens when behaviour goes wrong. Without budgets and a backstop, one confused loop turns a working demo into an unbounded bill. Reliability is 30% of the rubric for exactly this reason.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.